In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy.special import gammaln

plt.rcParams.update({
    "text.usetex":        False,
    "font.family":        "serif",
    "font.serif":         ["Palatino", "Palatino Linotype", "Georgia",
                           "Times New Roman", "DejaVu Serif"],
    "mathtext.fontset":   "cm",
    "axes.labelsize":     9,
    "axes.titlesize":     9,
    "xtick.labelsize":    8,
    "ytick.labelsize":    8,
    "figure.dpi":         150,
})

TEAL = ( 30/255, 130/255, 130/255)
SAND = (184/255, 146/255,  58/255)
RED  = (177/255,  56/255,  69/255)

Q_VALUES = [0.25, 0.50, 0.75]
Q_COLORS = [TEAL, SAND, RED]
P_VALUES = [0.80, 0.95, 0.99]

def _bar_color(rgb, x, x_max, pale_mix=0.72):
    t = min(abs(x) / x_max, 1.0) if x_max > 0 else 0.0
    return tuple(c + (1 - c) * pale_mix * t for c in rgb)

def _edge_color(rgb, x, x_max, pale_mix=0.35):
    t = min(abs(x) / x_max, 1.0) if x_max > 0 else 0.0
    return tuple(c + (1 - c) * pale_mix * t for c in rgb)

def simulate_erw_batch(n, p, q, n_sims, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    xi = np.zeros((n_sims, n + 1), dtype=np.int8)
    xi[:, 1] = np.where(rng.random(n_sims) < q, 1, -1)
    for t in range(1, n):
        idx  = rng.integers(1, t + 1, size=n_sims)
        past = xi[np.arange(n_sims), idx]
        flip = np.where(rng.random(n_sims) < p, 1, -1)
        xi[:, t + 1] = past * flip
    return xi[:, 1:].sum(axis=1).astype(float)

N_STEPS = 5_000
N_SIMS  = 15_000
N_BINS  = 52

rng = np.random.default_rng(2026)

finals = {}
for p_val in P_VALUES:
    alpha = 2 * p_val - 1
    scale = np.exp(alpha * np.log(N_STEPS) - gammaln(alpha + 1))
    for q_val in Q_VALUES:
        beta = 2 * q_val - 1
        raw  = simulate_erw_batch(N_STEPS, p=p_val, q=q_val, n_sims=N_SIMS, rng=rng)
        finals[(p_val, q_val)] = raw / scale - beta

row_xlim = {}
row_ylim = {}
row_bin_edges = {}
row_counts = {}

for p_val in P_VALUES:

    all_abs = np.concatenate([np.abs(finals[(p_val, q)]) for q in Q_VALUES])
    xlim = float(np.quantile(all_abs, 0.995)) * 1.05

    xlim = np.ceil(xlim * 10) / 10
    row_xlim[p_val] = xlim

    edges = np.linspace(-xlim, xlim, N_BINS + 1)
    row_bin_edges[p_val] = edges

    ymax = 0.0
    for q_val in Q_VALUES:
        c, _ = np.histogram(finals[(p_val, q_val)], bins=edges, density=True)
        row_counts[(p_val, q_val)] = c
        ymax = max(ymax, c.max())
    row_ylim[p_val] = ymax * 1.08

fig, axes = plt.subplots(3, 3, figsize=(7.2, 7.8))
fig.subplots_adjust(left=0.09, right=0.98, bottom=0.06, top=0.93,
                    wspace=0.28, hspace=0.85)

LABEL_GREY = (0.55, 0.55, 0.55)

def _nice_xticks(xlim):
    t = np.round(xlim * 0.75, 1)

    if t <= 0:
        t = np.round(xlim * 0.5, 1)
    return [-t, 0.0, t]

for i, p_val in enumerate(P_VALUES):
    xlim   = row_xlim[p_val]
    ymax   = row_ylim[p_val]
    edges  = row_bin_edges[p_val]
    width  = edges[1] - edges[0]
    centers = 0.5 * (edges[:-1] + edges[1:])
    xticks = _nice_xticks(xlim)

    for j, (q_val, color) in enumerate(zip(Q_VALUES, Q_COLORS)):
        ax = axes[i, j]
        counts = row_counts[(p_val, q_val)]

        for xc, h in zip(centers, counts):
            ax.bar(xc, h, width=width,
                   color=_bar_color(color, xc, xlim),
                   edgecolor=_edge_color(color, xc, xlim),
                   linewidth=0.28, alpha=0.70, zorder=2)

        ax.set_xlabel(r"$Z_n$", labelpad=2)
        ax.set_xlim(-xlim, xlim)
        ax.set_ylim(0, ymax)

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_linewidth(0.55)
        ax.spines["bottom"].set_linewidth(0.55)
        ax.tick_params(width=0.55, length=3, direction="out")

        ax.xaxis.set_major_locator(ticker.FixedLocator(xticks))
        def _fmt(t, _, ticks=xticks):

            if t == 0:
                return r"$0$"
            s = f"{t:g}"
            return fr"${s}$"
        ax.xaxis.set_major_formatter(ticker.FuncFormatter(_fmt))
        ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=3))

        if j == 0:
            ax.set_ylabel("Density", labelpad=4, fontsize=8)
        else:
            ax.tick_params(axis="y", labelleft=False)

P_LABEL_OFFSET = 0.040
Q_LABEL_OFFSET = 0.007

for i, p_val in enumerate(P_VALUES):
    row_top = axes[i, 1].get_position().y1

    fig.text(0.5, row_top + P_LABEL_OFFSET, fr"$p\!=\!{p_val:.2f}$",
             ha="center", va="bottom", fontsize=9.5, color=LABEL_GREY)

    for j, q_val in enumerate(Q_VALUES):
        col_x = 0.5 * (axes[i, j].get_position().x0 + axes[i, j].get_position().x1)
        fig.text(col_x, row_top + Q_LABEL_OFFSET, fr"$q\!=\!{q_val:.2f}$",
                 ha="center", va="bottom", fontsize=8.5, color=LABEL_GREY)

fig.savefig("figure_4_8.pdf", bbox_inches="tight", pad_inches=0.02)
print("Saved figure_4_8.pdf")